## Tony Grove - Precip Reanalysis Data Extraction
This script is used develop methods for downloading and adjusting NLDAS Reanalysis precipitation data. Tony Grove has multiple datasets to compare.

The goal is to adjust the reanalysis precipitation data to match the snotel daily cumulative value. Also, precip must be partitioned between rain and snow.

In [22]:
%matplotlib ipympl
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from gdriver.gdriver import get_file, get_credentials, getCSVtoPandas
from io import StringIO

#### Load SNOTEL data
Tony Grove snotel data should be in file: bcqc_41.89000_-111.57000.txt

The snotel data is QC'd from raw daily measurements, which are going to correspond to the local timezone. 

In [2]:
# Load gdriver credentials
credentials = get_credentials()

In [ ]:
# Download the txt file from Google Drive
rawfile = get_file('bcqc_41.89000_-111.57000.txt',creds=credentials)

# Read the txt file into a pandas dataframe. No header, separator is spaces
snotel_data = pd.read_csv(rawfile, delimiter=r'\s+', header=None, names=['Year', 'Month', 'Day', 'Precipitation', 'Max Temp', 'Min Temp', 'Avg Temp', 'Snow Water Equivalent'])

In [ ]:
snotel_data.head()

In [5]:
# convert Month, Day, Year to a datetime object and set as index
snotel_data['Date'] = pd.to_datetime(snotel_data[['Year', 'Month', 'Day']])
snotel_data.set_index('Date', inplace=True)

In [33]:
# Generate Avg Temp in C from F
snotel_data['Avg Temp C'] = (snotel_data['Avg Temp'] - 32) * 5/9

### Load Reanalysis Data
Using NLDAS which has an hourly timestep and 1/8 degree resolution. The TG Snotel data starts 2009-10-01 and goes thru September 2021. I only need to model thru the 2021 melt season, so will download up thru May 2021.

** Note: Starting with just one year of data for testing


In [12]:
# Define the base URL for the NLDAS Data Rods API
base_url = 'https://hydro1.gesdisc.eosdis.nasa.gov/daac-bin/access/timeseries.cgi'

# Define the parameters for the API request
params = {
    'variable': 'NLDAS2:NLDAS_FORA0125_H_v2.0:Tair',  # Example variable: 2-meter temperature
    'startDate': '2015-10-01T00',  # Start date in YYYY-MM-DDTHH format
    'endDate': '2016-10-01T00',    # End date in YYYY-MM-DDTHH format
    'location': 'GEOM:POINT(-111.57, 41.89)',  # Location as a point (longitude, latitude), note that the space in lat is important!!
    'type': 'asc2'  # Response format (e.g., asc2 for ASCII)
}

In [13]:
# Make the GET request to the API
response = requests.get(base_url, params=params)

# Check if the request was successful
if response.status_code == 200:
    # Print the response content
    nldas_csv = StringIO(response.text)
else:
    print(f"Error: {response.status_code}")

It took about 40sec to download about a year of data. By accident I only downloaded one day of data and it still took about 40sec, so it must be processing on the backend regardless of file size.

In [ ]:
# Review the first few lines of the data to determine how many rows to skip for the header
sample_lines = nldas_csv.getvalue().split('\n')[:15]
for line in sample_lines:
    print(line)

I reviewed the location shown in the metadata corresponding to the middle of the NLDAS grid cell. It is centered near the Franklin climate station and looks like it covers much of Beaver mountain. This may explain why the temperatures are so low relative to the TG data.

In [15]:
# Based on the inspection, I should skip the first 12 rows of data
skip_rows = 12

# Load the data into a Pandas DataFrame, skipping the necessary rows
nldas_csv.seek(0)  # Reset the StringIO object to the beginning
nldas_data = pd.read_csv(nldas_csv, delimiter=r'\s+', skiprows=skip_rows, index_col=0, parse_dates=True)

In [16]:
# Convert the temperature from Kelvin to Celsius
nldas_data['Tair'] = nldas_data['Data'] - 273.15

### Download GAMUT temperature data

In [ ]:
# Get the ST110 data
gamut_data = getCSVtoPandas('TonyGrove/LR_TG_C_AirTemp_ST110_Avg_SourceID_1_QC_1.csv',credentials,skiprows=56)

# Change index to DateTimeUTC
gamut_data['DateTimeUTC'] = pd.to_datetime(gamut_data['DateTimeUTC'])
gamut_data = gamut_data.set_index('DateTimeUTC')

# Convert -9999 to NaN
gamut_data = gamut_data.replace(-9999, np.nan)

### Timezone Adjustment
The snotel data is daily, QC'd from daily values. It isn't clear what timezone the datalogger calculating the daily values is using... probably local?

The reanalysis data must be in UTC since the input date range could be applied to anywhere in the model domain.

I will convert timezones to MST and make some comparisons.

In [28]:
nldas_data.index = nldas_data.index.tz_localize('UTC')
gamut_data.index = gamut_data.index.tz_localize('UTC')

In [29]:
# Convert to America/Phoenix timezone since this is essentially MST with no daylight savings
nldas_data = nldas_data.tz_convert('America/Phoenix')
gamut_data = gamut_data.tz_convert('America/Phoenix')

### Compare temperatures between reanalysis and observed

In [ ]:
# Compare Dec 2015
fig, ax = plt.subplots(figsize=(15, 6))
nldas_data['Tair']['2015-12'].plot(ax=ax, label='NLDAS', color='g')
gamut_data['AirTemp_ST110_Avg']['2015-12'].plot(ax=ax, label='ST110', color='b')
snotel_data['Avg Temp C']['2015-12'].plot(ax=ax, style='r*', label='SNOTEL')

In [ ]:
axs = nldas_data['Tair'].plot(title='NLDAS 2-meter Temperature', xlabel='Date', ylabel='Temperature (°C)')
snotel_data['Avg Temp']['2009-10-01':'2010-10-01'].plot(ax=axs, style='r*')

### Downscaling of NLDAS
My approach here will be to resample NLDAS to daily mean and subtract it from the observed daily means. Then shift the reanalysis by the average of the offsets.

In [21]:
# Extract just a year of TG data for initial testing
snotel_data_2009 = snotel_data['2009-10-01':'2010-10-01']

In [20]:
# Resample the NLDAS data to daily values
nldas_daily = nldas_data.resample('D').mean()

In [ ]:
# Calculate offset between NLDAS and TG data
offset = snotel_data_2009['Avg Temp'] - nldas_daily['Tair']
mean_offset = offset.mean()
print(f"Mean offset: {mean_offset:.2f} °C")

In [24]:
nldas_data['Tair_corrected'] = nldas_data['Tair'] + mean_offset

In [ ]:
axs2 = nldas_data['Tair_corrected'].plot(title='NLDAS 2-meter Temperature', xlabel='Date', ylabel='Temperature (°C)')
snotel_data['Avg Temp']['2009-10-01':'2010-10-01'].plot(ax=axs2, style='r*')